# 🧠 GhostFaceNet → TFLite Conversion Pipeline v2
### AMS (Attendance Management System) — Android Offline Deployment

---

| Step | What Happens |
|---|---|
| 1 | Install all dependencies |
| 2 | **Auto-download** GhostFaceNet `.h5` (HuggingFace → DeepFace fallback) |
| 3 | Verify file is real HDF5 binary (not corrupted HTML redirect) |
| 4 | Load model + model surgery (fix grouped Conv2D, GELU, extract_patches) |
| 5 | Convert to 4 TFLite variants (float32, float16, dynamic_int8, full_int8) |
| 6 | Verify accuracy of all variants vs original Keras model |
| 7 | AMS integration test (enrollment + recognition simulation) |
| 8 | Final summary + React Native integration guide |

> **Run all cells top to bottom. No manual file download needed.**

---
## 📦 Step 1 — Install Dependencies

In [ ]:
# Uninstall conflicting keras versions first
!pip uninstall -y keras keras-core keras-nightly 2>/dev/null

# Core packages
!pip install -q tensorflow==2.12.0
!pip install -q tf-keras==2.12.0
!pip install -q keras==2.12.0

# GhostFaceNet backbone + model surgery
!pip install -q keras-cv-attention-models

# Auto-download utilities
!pip install -q huggingface_hub
!pip install -q deepface
!pip install -q gdown

# Utilities
!pip install -q numpy Pillow matplotlib scikit-learn tqdm

print("\n✅ All packages installed — restart runtime if on Colab then re-run from Step 2")

---
## 🔧 Step 2 — Imports & Version Check

In [ ]:
import os, sys, glob, random, shutil
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from tqdm import tqdm

# Force TF 2.x Keras
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import tensorflow as tf
from tensorflow import keras

print(f"Python     : {sys.version.split()[0]}")
print(f"TensorFlow : {tf.__version__}")
print(f"Keras      : {keras.__version__}")
print(f"NumPy      : {np.__version__}")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"\n🚀 GPU : {gpus[0].name}")
    tf.config.experimental.set_memory_growth(gpus[0], True)
else:
    print("\n⚠️  No GPU — CPU mode (conversion still works fine)")

---
## ⚙️ Step 3 — Configuration

In [ ]:
# ─── OUTPUT ───────────────────────────────────────────────────────────────────
OUTPUT_DIR        = './tflite_output'
WEIGHTS_DIR       = './weights'

# ─── MODEL ────────────────────────────────────────────────────────────────────
# Options:
#   'GhostFaceNet_W1.3_S1_ArcFace.h5'    ← 99.78% LFW — RECOMMENDED
#   'GhostFaceNet_W1.3_S2_ArcFace.h5'    ← 99.71% LFW
#   'GhostFaceNet_W1.3_S1_CosFace.h5'    ← 99.75% LFW
MODEL_FILENAME    = 'GhostFaceNet_W1.3_S1_ArcFace.h5'

# ─── INPUT SPEC (GhostFaceNet standard) ───────────────────────────────────────
INPUT_H           = 112
INPUT_W           = 112
INPUT_C           = 3
EMB_DIM           = 512

# ─── INT8 CALIBRATION ─────────────────────────────────────────────────────────
# Point to any folder with face images for full INT8 quantization
# Leave as None to use random noise (still works, slightly less optimal)
CALIB_IMAGE_DIR   = None   # e.g. '/content/lfw_sample'
CALIB_COUNT       = 100

# ─── Create dirs ──────────────────────────────────────────────────────────────
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(WEIGHTS_DIR, exist_ok=True)

MODEL_H5_PATH = os.path.join(WEIGHTS_DIR, MODEL_FILENAME)

print(f"📁 Output dir  : {os.path.abspath(OUTPUT_DIR)}")
print(f"📁 Weights dir : {os.path.abspath(WEIGHTS_DIR)}")
print(f"🎯 Target model: {MODEL_FILENAME}")
print(f"📐 Input shape : ({INPUT_H}, {INPUT_W}, {INPUT_C})")
print(f"📊 Embedding   : {EMB_DIM}-dim")

---
## ⬇️ Step 4 — Auto-Download GhostFaceNet `.h5`
Tries **HuggingFace Hub** first (most reliable), then **DeepFace**, then **gdown** as fallbacks.
No manual download needed.

In [ ]:
def is_valid_hdf5(path):
    """Check if file is a real HDF5 binary, not an HTML redirect page."""
    try:
        with open(path, 'rb') as f:
            sig = f.read(8)
        return sig == b'\x89HDF\r\n\x1a\n'
    except Exception:
        return False


def download_via_huggingface(dest_path):
    """Download from HuggingFace Hub — serengil/deepface_models repo."""
    print("🔵 Trying HuggingFace Hub...")
    try:
        from huggingface_hub import hf_hub_download
        path = hf_hub_download(
            repo_id='serengil/deepface_models',
            filename=MODEL_FILENAME,
            local_dir=WEIGHTS_DIR,
            force_download=False,
        )
        if is_valid_hdf5(path):
            if path != dest_path:
                shutil.copy(path, dest_path)
            print(f"   ✅ HuggingFace: downloaded → {dest_path}")
            return True
        else:
            print("   ❌ HuggingFace: file corrupted (not HDF5)")
            return False
    except Exception as e:
        print(f"   ❌ HuggingFace failed: {e}")
        return False


def download_via_deepface(dest_path):
    """Use DeepFace's internal downloader — handles URL + integrity automatically."""
    print("🟡 Trying DeepFace downloader...")
    try:
        # DeepFace stores weights in ~/.deepface/weights/
        deepface_weights = os.path.join(
            os.path.expanduser('~'), '.deepface', 'weights', MODEL_FILENAME
        )
        if is_valid_hdf5(deepface_weights):
            shutil.copy(deepface_weights, dest_path)
            print(f"   ✅ DeepFace cache hit → {dest_path}")
            return True

        # Trigger DeepFace to download it
        from deepface import DeepFace
        # Use a dummy image — we just want the download to trigger
        dummy = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
        dummy_path = '/tmp/dummy_face.png'
        Image.fromarray(dummy).save(dummy_path)

        try:
            DeepFace.represent(
                img_path=dummy_path,
                model_name='GhostFaceNet',
                detector_backend='skip',
                enforce_detection=False,
            )
        except Exception:
            pass  # We only care about the download side-effect

        if is_valid_hdf5(deepface_weights):
            shutil.copy(deepface_weights, dest_path)
            print(f"   ✅ DeepFace: downloaded → {dest_path}")
            return True
        else:
            print("   ❌ DeepFace: file corrupted or not downloaded")
            return False
    except Exception as e:
        print(f"   ❌ DeepFace failed: {e}")
        return False


def download_via_gdown(dest_path):
    """
    gdown with fuzzy=True bypasses Google Drive virus-scan HTML redirect.
    File IDs from: https://github.com/HamadYA/GhostFaceNets README.
    """
    print("🟠 Trying gdown (Google Drive)...")
    # Known Google Drive file IDs for GhostFaceNet weights
    GDRIVE_IDS = {
        'GhostFaceNet_W1.3_S1_ArcFace.h5' : '1UoaMLae1h1B9B7a8lhXP6Cg4RKSz7s3n',
        'GhostFaceNet_W1.3_S2_ArcFace.h5' : '1xM-KBiT_vIFiVMgLWOtBUxPD8EoZWfZd',
        'GhostFaceNet_W1.3_S1_CosFace.h5' : '1x2tGpyIbFT79NjIMdaLhf8ukmAnrUXf5',
    }
    file_id = GDRIVE_IDS.get(MODEL_FILENAME)
    if not file_id:
        print(f"   ❌ No known Google Drive ID for {MODEL_FILENAME}")
        print(f"   → Get the ID from: https://github.com/HamadYA/GhostFaceNets")
        return False
    try:
        import gdown
        url = f'https://drive.google.com/uc?id={file_id}'
        gdown.download(url, dest_path, fuzzy=True, quiet=False)
        if is_valid_hdf5(dest_path):
            print(f"   ✅ gdown: downloaded → {dest_path}")
            return True
        else:
            print("   ❌ gdown: file corrupted (HTML redirect — try manual download)")
            return False
    except Exception as e:
        print(f"   ❌ gdown failed: {e}")
        return False


# ─── Main download orchestrator ────────────────────────────────────────────────
print(f"🔍 Checking for existing model at: {MODEL_H5_PATH}")

if is_valid_hdf5(MODEL_H5_PATH):
    size_mb = os.path.getsize(MODEL_H5_PATH) / (1024 * 1024)
    print(f"✅ Valid model already exists ({size_mb:.1f} MB) — skipping download")
else:
    if os.path.exists(MODEL_H5_PATH):
        print(f"⚠️  File exists but is NOT valid HDF5 (likely HTML redirect) — re-downloading")
        os.remove(MODEL_H5_PATH)

    print(f"\n⬇️  Downloading {MODEL_FILENAME}...\n")
    success = (
        download_via_huggingface(MODEL_H5_PATH) or
        download_via_deepface(MODEL_H5_PATH) or
        download_via_gdown(MODEL_H5_PATH)
    )

    if not success:
        raise RuntimeError(
            f"\n❌ All download methods failed for {MODEL_FILENAME}\n"
            f"Manual download steps:\n"
            f"  1. Go to: https://github.com/HamadYA/GhostFaceNets\n"
            f"  2. Find the Google Drive link for {MODEL_FILENAME}\n"
            f"  3. Download and place at: {os.path.abspath(MODEL_H5_PATH)}\n"
            f"  4. Re-run this cell"
        )

# ─── Final HDF5 validation ─────────────────────────────────────────────────────
assert is_valid_hdf5(MODEL_H5_PATH), (
    f"\n❌ {MODEL_H5_PATH} is not a valid HDF5 file!\n"
    f"Delete it manually and re-run this cell."
)
size_mb = os.path.getsize(MODEL_H5_PATH) / (1024 * 1024)
print(f"\n🎉 Model ready: {MODEL_H5_PATH} ({size_mb:.1f} MB)")

---
## 🔬 Step 5 — Load Model + Model Surgery
Fixes grouped Conv2D, GELU, and extract_patches before TFLite conversion.

In [ ]:
from keras_cv_attention_models import model_surgery

# ─── Load ─────────────────────────────────────────────────────────────────────
print(f"🔃 Loading model...")
model = tf.keras.models.load_model(MODEL_H5_PATH, compile=False)
print(f"✅ Loaded — Input: {model.input_shape}  Output: {model.output_shape}")
print(f"   Parameters: {model.count_params():,}")

# ─── Sanity check input/output shape ──────────────────────────────────────────
assert len(model.input_shape) == 4, "Expected 4D input (batch, H, W, C)"
_, h, w, c = model.input_shape
out_dim = model.output_shape[-1]
print(f"\n📐 Input : {h}×{w}×{c}")
print(f"📊 Output: {out_dim}-dim embedding")

if (h, w) != (INPUT_H, INPUT_W):
    print(f"⚠️  Shape mismatch — updating config: INPUT_H/W → {h}/{w}")
    INPUT_H, INPUT_W = h, w
if out_dim != EMB_DIM:
    print(f"⚠️  Embedding dim mismatch — updating config: EMB_DIM → {out_dim}")
    EMB_DIM = out_dim

# ─── Model Surgery ────────────────────────────────────────────────────────────
# Fixes three TFLite incompatibilities in GhostFaceNet:
#   1. Conv2D groups > 1      → split into standard Conv2D ops
#   2. GELU activation        → approximate GELU (TFLite-compatible)
#   3. tf.image.extract_patches → equivalent Conv2D
print("\n🔧 Applying model surgery for TFLite compatibility...")
try:
    model_tflite = model_surgery.prepare_for_tflite(model)
    print("   ✅ Groups Conv2D → Split Conv2D")
    print("   ✅ GELU → Approximate GELU")
    print("   ✅ extract_patches → Conv2D equivalent")
except Exception as e:
    print(f"   ⚠️  model_surgery.prepare_for_tflite failed: {e}")
    print("   → Falling back to original model (will use SELECT_TF_OPS)")
    model_tflite = model

# ─── Post-surgery accuracy check ──────────────────────────────────────────────
np.random.seed(0)
dummy = np.random.uniform(-1, 1, (1, INPUT_H, INPUT_W, INPUT_C)).astype(np.float32)
out_orig    = model.predict(dummy, verbose=0)
out_surgery = model_tflite.predict(dummy, verbose=0)
max_diff    = float(np.max(np.abs(out_orig - out_surgery)))
print(f"\n🔬 Post-surgery output diff (max): {max_diff:.2e}")
print("   ✅ Surgery preserved model accuracy" if max_diff < 1e-4 else f"   ⚠️  Larger diff — review surgery ({max_diff:.6f})")

---
## 🛠️ Step 6 — Conversion Helpers

In [ ]:
def preprocess(img_or_path, size=(112, 112)):
    """GhostFaceNet normalization: [0,255] → [-1,1]"""
    if isinstance(img_or_path, str):
        img = np.array(Image.open(img_or_path).convert('RGB').resize(size), dtype=np.float32)
    else:
        img = img_or_path.astype(np.float32)
    return np.expand_dims((img - 127.5) * 0.0078125, 0)  # (1,H,W,3)


def make_rep_dataset(image_dir=None, count=100):
    """Representative dataset generator for full INT8 quantization."""
    if image_dir and os.path.isdir(image_dir):
        paths = []
        for ext in ('*.jpg','*.jpeg','*.png'):
            paths += glob.glob(os.path.join(image_dir,'**',ext), recursive=True)
        random.shuffle(paths)
        paths = paths[:count]
        print(f"   📸 Calibration: {len(paths)} real face images from {image_dir}")
        def gen():
            for p in paths:
                try: yield [preprocess(p, (INPUT_H, INPUT_W))]
                except: pass
    else:
        print("   📸 Calibration: random noise (no CALIB_IMAGE_DIR set)")
        def gen():
            for _ in range(count):
                yield [np.random.uniform(-1, 1, (1, INPUT_H, INPUT_W, INPUT_C)).astype(np.float32)]
    return gen


def run_tflite(tflite_path, inp):
    """Run inference on any TFLite model — handles float32, float16, int8 automatically."""
    interp = tf.lite.Interpreter(model_path=tflite_path)
    interp.allocate_tensors()
    in_det  = interp.get_input_details()[0]
    out_det = interp.get_output_details()[0]

    dtype = in_det['dtype']
    if dtype in (np.int8, np.uint8):
        s, z = in_det['quantization']
        data = np.clip(inp / s + z, np.iinfo(dtype).min, np.iinfo(dtype).max).astype(dtype)
    else:
        data = inp.astype(np.float32)

    interp.set_tensor(in_det['index'], data)
    interp.invoke()
    out = interp.get_tensor(out_det['index'])

    if out_det['dtype'] in (np.int8, np.uint8):
        s, z = out_det['quantization']
        out = (out.astype(np.float32) - z) * s
    return out


def cosine_sim(a, b):
    a, b = a.flatten(), b.flatten()
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))


def make_converter(model, quant_type, rep_dataset_fn=None):
    """Build a TFLiteConverter with correct settings per quantization type."""
    conv = tf.lite.TFLiteConverter.from_keras_model(model)
    conv.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS,
        tf.lite.OpsSet.SELECT_TF_OPS,
    ]
    conv._experimental_lower_tensor_list_ops = False

    if quant_type == 'float32':
        pass  # No quantization

    elif quant_type == 'float16':
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        conv.target_spec.supported_types = [tf.float16]

    elif quant_type == 'dynamic_int8':
        # Weights→INT8, activations dynamic at runtime, no calib needed
        conv.optimizations = [tf.lite.Optimize.DEFAULT]

    elif quant_type == 'full_int8':
        # Both weights + activations → INT8, enables NNAPI on old Android
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        conv.representative_dataset = rep_dataset_fn
        conv.target_spec.supported_ops = [
            tf.lite.OpsSet.TFLITE_BUILTINS_INT8,
            tf.lite.OpsSet.TFLITE_BUILTINS,
            tf.lite.OpsSet.SELECT_TF_OPS,
        ]
        # Keep IO as float32 → easier React Native integration
        conv.inference_input_type  = tf.float32
        conv.inference_output_type = tf.float32

    return conv


print("✅ Helper functions ready")

---
## 🔄 Step 7 — Convert All 4 TFLite Variants

In [ ]:
VARIANTS = [
    ('float32',      'ghostfacenet_float32.tflite',      None),
    ('float16',      'ghostfacenet_float16.tflite',      None),
    ('dynamic_int8', 'ghostfacenet_dynamic_int8.tflite', None),
    ('full_int8',    'ghostfacenet_full_int8.tflite',    make_rep_dataset(CALIB_IMAGE_DIR, CALIB_COUNT)),
]

conversion_results = {}

for quant_type, filename, rep_fn in VARIANTS:
    out_path = os.path.join(OUTPUT_DIR, filename)
    print(f"\n{'─'*50}")
    print(f"🔄 [{quant_type}] Converting...")

    try:
        conv = make_converter(model_tflite, quant_type, rep_fn)
        tflite_model = conv.convert()
        with open(out_path, 'wb') as f:
            f.write(tflite_model)
        size_mb = os.path.getsize(out_path) / (1024 * 1024)
        conversion_results[quant_type] = {'path': out_path, 'size': size_mb, 'success': True}
        print(f"   ✅ Saved → {filename} ({size_mb:.2f} MB)")
    except Exception as e:
        print(f"   ❌ Failed: {e}")
        conversion_results[quant_type] = {'path': out_path, 'size': 0, 'success': False, 'error': str(e)}

print(f"\n\n{'='*50}")
print(f" Conversion Complete — {sum(v['success'] for v in conversion_results.values())}/{len(VARIANTS)} succeeded")
print(f"{'='*50}")

---
## ✅ Step 8 — Accuracy Verification

In [ ]:
np.random.seed(42)
# Simulate a real face: random pixel values [0,255], then normalize
test_img  = np.random.randint(0, 255, (INPUT_H, INPUT_W, INPUT_C)).astype(np.float32)
test_norm = preprocess(test_img, (INPUT_H, INPUT_W))

# Ground truth from original Keras model (pre-surgery)
keras_emb = model.predict(test_norm, verbose=0)
print(f"Keras embedding — shape: {keras_emb.shape} | norm: {np.linalg.norm(keras_emb):.4f}\n")

verify_results = []
print(f"{'Variant':<22} {'Cosine Sim':>12} {'L2 Diff':>10} {'Size MB':>10} {'Status'}")
print('─' * 65)

for quant_type, info in conversion_results.items():
    if not info['success']:
        print(f"{quant_type:<22} {'SKIPPED (conversion failed)':>44}")
        continue
    try:
        tflite_emb = run_tflite(info['path'], test_norm)
        sim  = cosine_sim(keras_emb, tflite_emb)
        l2   = float(np.linalg.norm(keras_emb - tflite_emb))
        flag = '🟢' if sim > 0.999 else ('🟡' if sim > 0.990 else '🔴')
        print(f"{quant_type:<22} {sim:>12.6f} {l2:>10.6f} {info['size']:>9.2f}MB  {flag}")
        verify_results.append({'name': quant_type, 'sim': sim, 'l2': l2, 'size': info['size']})
    except Exception as e:
        print(f"{quant_type:<22} ERROR: {e}")

print('─' * 65)
print("🟢 >0.999 excellent  |  🟡 >0.990 acceptable  |  🔴 review needed")

---
## 📊 Step 9 — Visualize Results

In [ ]:
if verify_results:
    names  = [r['name'] for r in verify_results]
    sims   = [r['sim']  for r in verify_results]
    sizes  = [r['size'] for r in verify_results]
    colors = ['#10B981' if s > 0.999 else '#F59E0B' if s > 0.990 else '#EF4444' for s in sims]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor('#F8F7FF')
    fig.suptitle('GhostFaceNet → TFLite Conversion Results', fontsize=14, fontweight='bold', color='#1E1B4B')

    # Accuracy
    bars = ax1.bar(names, sims, color=colors, edgecolor='white', linewidth=1.5, zorder=3)
    ax1.set_facecolor('#F8F7FF')
    ax1.set_title('Accuracy vs Keras (Cosine Similarity)', fontweight='bold', color='#1E1B4B')
    ax1.set_ylabel('Cosine Similarity', color='#1E1B4B')
    ax1.set_ylim([min(sims) - 0.005, 1.001])
    ax1.axhline(0.999, color='#10B981', ls='--', alpha=0.6, label='Excellent (0.999)')
    ax1.axhline(0.990, color='#F59E0B', ls='--', alpha=0.6, label='Acceptable (0.990)')
    ax1.legend(fontsize=8)
    ax1.grid(axis='y', alpha=0.3, zorder=0)
    for bar, val in zip(bars, sims):
        ax1.text(bar.get_x() + bar.get_width()/2, val + 0.0003,
                 f'{val:.5f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

    # Size
    bars2 = ax2.bar(names, sizes, color='#4F46E5', edgecolor='white', linewidth=1.5, zorder=3)
    ax2.set_facecolor('#F8F7FF')
    ax2.set_title('Model Size (MB)', fontweight='bold', color='#1E1B4B')
    ax2.set_ylabel('Size (MB)', color='#1E1B4B')
    ax2.grid(axis='y', alpha=0.3, zorder=0)
    for bar, val in zip(bars2, sizes):
        ax2.text(bar.get_x() + bar.get_width()/2, val + 0.1,
                 f'{val:.1f} MB', ha='center', va='bottom', fontsize=9, fontweight='bold', color='#1E1B4B')

    plt.tight_layout()
    chart_path = os.path.join(OUTPUT_DIR, 'results.png')
    plt.savefig(chart_path, dpi=150, bbox_inches='tight', facecolor='#F8F7FF')
    plt.show()
    print(f"\n📊 Chart saved → {chart_path}")

---
## 📱 Step 10 — AMS Integration Test
Simulates your app's full enrollment + recognition pipeline.

In [ ]:
# Use best available model: dynamic_int8 preferred (best for old Android)
PRIORITY = ['dynamic_int8', 'full_int8', 'float16', 'float32']
test_model_path = next(
    (conversion_results[q]['path'] for q in PRIORITY if conversion_results.get(q, {}).get('success')),
    None
)
if not test_model_path:
    raise RuntimeError("No successful TFLite model found to test with")

print(f"🏢 AMS Integration Test")
print(f"   Using model: {os.path.basename(test_model_path)}\n")

THRESHOLD = 0.60
N_EMPLOYEES = 3
EMBEDDINGS_PER_EMPLOYEE = 5

np.random.seed(99)

# ── Enrollment ─────────────────────────────────────────────────────────────────
print("📋 Enrolling employees (5 embeddings each)...")
employees = {}
for eid in range(N_EMPLOYEES):
    base = np.random.uniform(0, 255, (INPUT_H, INPUT_W, INPUT_C)).astype(np.float32)
    embs = []
    for _ in range(EMBEDDINGS_PER_EMPLOYEE):
        noisy = np.clip(base + np.random.normal(0, 5, base.shape), 0, 255)
        emb = run_tflite(test_model_path, preprocess(noisy)).flatten()
        emb /= np.linalg.norm(emb) + 1e-10
        embs.append(emb)
    centroid = np.mean(embs, axis=0)
    centroid /= np.linalg.norm(centroid) + 1e-10
    employees[f'emp_{eid:03d}'] = {'embeddings': embs, 'centroid': centroid, 'base': base}
    print(f"   ✅ emp_{eid:03d} enrolled — {EMBEDDINGS_PER_EMPLOYEE} embeddings captured")

# ── Recognition (max-sim voting) ───────────────────────────────────────────────
print("\n🔍 Recognition test (max-sim voting, 3-frame confirmation sim)...\n")
all_pass = True
for target_id, data in employees.items():
    correct_count = 0
    for frame in range(3):  # Simulate 3-frame confirmation
        live = np.clip(data['base'] + np.random.normal(0, 8, data['base'].shape), 0, 255)
        live_emb = run_tflite(test_model_path, preprocess(live)).flatten()
        live_emb /= np.linalg.norm(live_emb) + 1e-10

        scores = {
            eid: max(cosine_sim(live_emb, e) for e in edata['embeddings'])
            for eid, edata in employees.items()
        }
        best_id    = max(scores, key=scores.get)
        best_score = scores[best_id]

        if best_id == target_id and best_score >= THRESHOLD:
            correct_count += 1

    confirmed = correct_count == 3
    status    = '✅' if confirmed else '❌'
    all_pass  = all_pass and confirmed
    print(f"   {status} {target_id} → confirmed: {confirmed} ({correct_count}/3 frames passed, score: {best_score:.4f})")

print(f"\n{'🎉 All employees recognised correctly!' if all_pass else '⚠️  Some recognitions failed — review threshold'}")

---
## 📋 Step 11 — Final Summary

In [ ]:
print("\n" + "═"*62)
print("  ✅  GHOSTFACENET → TFLITE CONVERSION COMPLETE")
print("═"*62)

LABELS = {
    'float32':      ('Float32  (baseline)',   'High-end Android API 28+'),
    'float16':      ('Float16  (half prec)',  'Mid-range Android API 26+'),
    'dynamic_int8': ('Dynamic INT8',          '✅  RECOMMENDED — Any Android API 21+'),
    'full_int8':    ('Full INT8  + NNAPI',    'Best perf on old chipsets'),
}

for qt, (label, usecase) in LABELS.items():
    info = conversion_results.get(qt, {})
    if info.get('success'):
        print(f"  {label:25s}  {info['size']:5.1f} MB  →  {usecase}")
    else:
        print(f"  {label:25s}  FAILED  →  {info.get('error','unknown error')[:40]}")

print("═"*62)
print()
print("📱 FOR YOUR REACT NATIVE AMS APP:")
print(f"   Copy   →  android/app/src/main/assets/ghostfacenet.tflite")
print(f"   Use    →  ghostfacenet_dynamic_int8.tflite")
print()
print("🔧 Changes vs your old MobileFaceNet config:")
print(f"   inputSize    : 192  →  {INPUT_H}  (update resize plugin call)")
print(f"   embeddingDim : 192  →  {EMB_DIM}  (update cosine similarity arrays)")
print(f"   normalization: same →  (pixel - 127.5) * 0.0078125")
print(f"   runSync API  : same →  model.runSync([input])")
print(f"   threshold    : same →  global 0.60, per-employee cap 0.72")
print()
print("📁 config.ts update:")
print(f"   inputSize: {INPUT_H},  // was 192")
print(f"   embeddingDim: {EMB_DIM},  // was 192")
print(f"   modelFile: 'ghostfacenet.tflite'")
print()
print(f"📂 All files in: {os.path.abspath(OUTPUT_DIR)}")
print("═"*62)